# Experimento 1 — Regressão Logística sem balanceamento utilizando 4 variáveis

Este experimento introduz a Regressão Logística como uma *baseline* linear para o dataset de qualidade hídrica, mantendo a base de comparação com os experimentos anteriores (Random Forest e LightGBM).

Diferente dos algoritmos baseados em árvores, a Regressão Logística assume que as fronteiras de decisão entre as classes (Excelente, Boa, Atenção, Crítica) podem ser separadas de forma linear. Além disso, por ser um modelo dependente de cálculos de distância e otimização de gradiente, ele é altamente sensível à escala das variáveis.

Por conta dessa lacuna em relação aos modelos anteriores, foi introduzida uma etapa de padronização (`StandardScaler`) no pipeline para as variáveis contínuas.

As variáveis utilizadas como entrada foram:
- `Temperature (cel)` (Numérica - Padronizada)
- `Orthophosphate (mg/l)` (Numérica - Padronizada)
- `Country` (Categórica - One-Hot Encoded)
- `Waterbody Type` (Categórica - One-Hot Encoded)

A variável alvo (`y`) permanece:
- `conama_status`

Nenhuma técnica de balanceamento foi aplicada. O objetivo é avaliar a capacidade de um modelo linear simples lidar com o desbalanceamento natural dos dados e comparar sua capacidade de generalização e *overfitting* em relação aos modelos de ensemble testados anteriormente.

## Preparação do ambiente

In [ ]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/dataset_pisi3/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [ ]:
# PREPARANDO O AMBIENTE PARA MACHINE LEARNING
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# DEFINIÇÃO DE X E Y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

In [ ]:
# TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 4)
Teste: (28280, 4)


In [ ]:
# PRÉ-PROCESSAMENTO COM STANDARD SCALER PARA VARIÁVEIS NUMÉRICAS
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [ ]:
# CONSTRUÇÃO DO PIPELINE
# max_iter=1000 adicionado para garantir a convergência do algoritmo
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=SEED,
                max_iter=1000,
                multi_class='multinomial'
            )
        )
    ]
)

In [ ]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type']),
                                                 ('num', StandardScaler(),
                                                  ['Temperature (cel)',
                                                   'Orthophosphate (mg/l)'])])),
                ('classifier',
                 LogisticRegression(max_iter=1000, multi_class='multinomial',
                                    random_state=42))])

## Avaliação das Métricas de Treino

Assim como nos experimentos anteriores, avaliaremos o conjunto de treino para investigar possíveis ocorrências de *overfitting* ou, neste caso, *underfitting* (comum quando modelos lineares tentam explicar fenômenos altamente complexos).

In [ ]:
y_train_pred = model.predict(X_train)

In [ ]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted',
    zero_division=0
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted',
    zero_division=0
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted',
    zero_division=0
)

train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:\n", train_accuracy)
print("\nTrain Precision:\n", train_precision)
print("\nTrain Recall:\n", train_recall)
print("\nTrain F1:\n", train_f1)
print("\nTrain Confusion Matrix:\n", train_confusion_matrix)

Train Accuracy:
 0.7368877023311734

Train Precision:
 0.6513255813274198

Train Recall:
 0.7368877023311734

Train F1:
 0.6642099418704254

Train Confusion Matrix:
 [[ 1096  1473     4  4987]
 [  713  1587     3 19375]
 [  165   393     3   545]
 [  507  1596     2 80670]]


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# Teste com 4 variáveis - Regressão Logística sem balanceamento
print("Accuracy:\n", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy:
 0.7376591230551627

Classification Report:
               precision    recall  f1-score   support

     Atenção       0.47      0.15      0.23      1890
         Boa       0.33      0.08      0.13      5419
     Crítica       0.00      0.00      0.00       277
   Excelente       0.76      0.97      0.86     20694

    accuracy                           0.74     28280
   macro avg       0.39      0.30      0.30     28280
weighted avg       0.65      0.74      0.67     28280


Confusion Matrix:
 [[  286   348     1  1255]
 [  170   443     0  4806]
 [   39   108     0   130]
 [  120   440     2 20132]]


In [ ]:
# Salvamento das métricas
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

resultados = {
    "experimento": "exp_01_logistic_regression_sem_balanceamento",
    "algoritmo": "LogisticRegression",
    "balanceamento": "nenhum",
    "n_features": X.shape[1],
    "accuracy_treino": round(train_accuracy, 4),
    "accuracy_teste": round(test_accuracy, 4),
    "f1_weighted_treino": round(train_f1, 4),
    "f1_weighted_teste": round(test_f1, 4),
}

pd.DataFrame([resultados]).to_csv(
    "/content/drive/MyDrive/dataset_pisi3/resultados/resultados_experimento_1.csv",
    mode="a",
    index=False,
    header=False
)

print("Métricas salvas com sucesso.")


Métricas salvas com sucesso.


## Análise dos Resultados

### Desempenho Geral

O modelo de Regressão Logística, mesmo com o pré-processamento de padronização das variáveis numéricas, apresentou uma acurácia de treino de `r round(train_accuracy, 4)` e uma acurácia de teste de `r round(test_accuracy, 4)`. O valor de F1-score ponderado foi de `r round(train_f1, 4)` no treino e `r round(test_f1, 4)` no teste.

Estes resultados indicam um desempenho moderado, com uma ligeira diferença entre treino e teste, sugerindo que o modelo não está sofrendo de *overfitting* severo. No entanto, a acurácia geral e o F1-score podem ser considerados baixos para um problema de classificação.

### Desempenho por Classe e Desbalanceamento

A análise do *Classification Report* e da *Confusion Matrix* é crucial, especialmente devido ao desbalanceamento das classes. Podemos observar que:

- **Classe 'Excelente':** O modelo apresenta um alto recall e precisão para a classe 'Excelente', que é a classe majoritária. Isso é esperado, pois o modelo tende a aprender a classificar a classe mais frequente com maior sucesso para otimizar a acurácia geral.
- **Classes Minoritárias ('Atenção', 'Boa', 'Crítica'):** O recall e a precisão para as classes minoritárias são significativamente baixos, ou até mesmo zero em alguns casos (como para a classe 'Crítica' no teste). Isso demonstra que o modelo está lutando para identificar corretamente as instâncias dessas classes. A baixa representatividade dessas classes no conjunto de dados, combinada com a natureza linear da Regressão Logística, dificulta a aprendizagem de padrões distintivos para elas.

### Comparação com Modelos de Ensemble (Contexto)

Conforme antecipado, a Regressão Logística, sendo um modelo linear mais simples, apresenta um desempenho inferior aos modelos baseados em árvores (como Random Forest e LightGBM) em problemas com relações não lineares complexas e dados desbalanceados. A incapacidade de modelar fronteiras de decisão não lineares e a sensibilidade ao desbalanceamento de classes são os principais fatores que contribuem para essa diferença.

### Próximos Passos

Os resultados deste experimento reforçam a necessidade de explorar:

1.  **Técnicas de Balanceamento de Classes:** Para dar mais peso às classes minoritárias e permitir que o modelo aprenda com elas de forma mais eficaz.
2.  **Adição de Novas Variáveis:** A inclusão de mais *features* relevantes pode fornecer informações adicionais que ajudem o modelo a distinguir melhor entre as classes.
3.  **Ajuste de Hiperparâmetros:** Otimizar os hiperparâmetros da Regressão Logística ou considerar outros modelos que sejam mais robustos ao desbalanceamento e capazes de capturar relações não lineares.